# trace-validity — one notebook, three stages

Does an **invalid reasoning trace** still reach a **correct answer**?
Goedel-Prover-SFT on FormalStep, judged by Lean 4 + Mathlib.

**Set `STAGE` in the first cell, then Runtime → Run all.**

| STAGE | runtime | what it does |
| --- | --- | --- |
| `generate` | **GPU** | samples proof trajectories → `results/traj_temp{T}.json` |
| `verify` | **CPU** | checks them in Lean → `results/results_temp{T}.json` |
| `analyze` | **CPU** | `results/summary.csv` + `figs/*.png` |
| `all` | GPU | all three on one box |

Generation needs a GPU; Lean verification does not. They are separate processes
that talk only through files in `results/`, so a timeout in one never destroys
the other's work.

**Every cell is safe to re-run.** Nothing already present in `results/` is
recomputed, so a disconnect means re-running the cell — not redoing the work.
Suggested order: GPU runtime with `STAGE="generate"`, then a CPU runtime with
`STAGE="verify"`, then `STAGE="analyze"`. Cell 5 carries `results/` through
Drive between the two runtimes.

## 1. Config

In [ ]:
# ---- the only cell you normally edit -------------------------------------
STAGE = "generate"          # "generate" | "verify" | "analyze" | "all"
TEMPS = [0, 0.2, 0.5, 0.8, 1.0]
N_QUESTIONS = 50            # first N rows of liuchengwu/FormalStep:train
N_TRAJ = 10                 # per question; forced to 1 at temp 0 (greedy)
# --------------------------------------------------------------------------

REPO_URL  = "https://github.com/Flashinl/trace-validity.git"
REPO_DIR  = "/content/trace-validity"
DRIVE_DIR = "/content/drive/MyDrive/trace-validity"
MATHLIB_DIR = "mathlib4"
MATHLIB_TARBALL = "mathlib4_build.tar.gz"

NEEDS_GPU  = STAGE in ("generate", "all")
NEEDS_LEAN = STAGE in ("verify", "all")

def fmt_temp(t):
    """Must match src/config.fmt_temp so we look for the right filenames."""
    return f"{float(t):g}"

print(f"STAGE={STAGE}  TEMPS={TEMPS}  N_QUESTIONS={N_QUESTIONS}  N_TRAJ={N_TRAJ}")
print(f"needs GPU: {NEEDS_GPU}   needs Lean: {NEEDS_LEAN}")

import shutil
if NEEDS_GPU and shutil.which("nvidia-smi") is None:
    print("\n*** WARNING: no GPU visible. Runtime > Change runtime type > GPU. ***")
if NEEDS_LEAN and not NEEDS_GPU and shutil.which("nvidia-smi") is not None:
    print("\nNote: verification is CPU-only. A GPU runtime here just burns quota.")

## 2. Clone the repo and install dependencies

The repo is **private**, so the clone needs a token. In Colab: click the 🔑 in
the left sidebar, add a secret named `GH_TOKEN` holding a GitHub fine-grained
PAT (read-only on this repo is enough for the clone; add write if you intend to
push results back from the notebook), and toggle notebook access on.

Two caveats about that token:

* It ends up in `.git/config` inside the Colab filesystem. Fine for a throwaway
  runtime with a read-only token — just don't screen-share that terminal.
* Commands are echoed with the token **masked**, because notebook outputs get
  committed and `!git clone https://$TOKEN@...` would write the PAT straight
  into the repo history.

Making the repo public removes this step entirely.

GPU dependencies (`torch` / `transformers` / `vllm`) are installed **only** when
`STAGE` needs a GPU — several minutes and a few GB of disk that the Lean runtime
has no use for. Re-running is cheap: the clone is skipped if the directory
exists, and pip no-ops on satisfied requirements.

In [ ]:
import os, re, subprocess, sys

IN_COLAB = "google.colab" in sys.modules

def _mask(cmd):
    """Strip any PAT before echoing -- notebook outputs get committed."""
    return re.sub(r"https://[^@/\s]+@", "https://***@", cmd)

def sh(cmd, check=True):
    print("$", _mask(cmd), flush=True)
    return subprocess.run(cmd, shell=True, check=check)

GH_TOKEN = None
if IN_COLAB:
    try:
        from google.colab import userdata
        GH_TOKEN = userdata.get("GH_TOKEN")
    except Exception as exc:
        print(f"no GH_TOKEN secret ({type(exc).__name__}) -- "
              "the clone will only work if the repo is public")

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        clone_url = REPO_URL
        if GH_TOKEN:
            clone_url = REPO_URL.replace("https://", f"https://{GH_TOKEN}@")
        sh(f"git clone {clone_url} {REPO_DIR}")
    else:
        print(f"{REPO_DIR} already cloned")
    os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

PY = sys.executable
sh(f"{PY} -m pip install -q -r requirements.txt")

if NEEDS_GPU:
    # torch / transformers / accelerate / vllm -- GPU runtime only.
    sh(f"{PY} -m pip install -q -r requirements-gpu.txt")
else:
    print("skipping requirements-gpu.txt (vllm/torch): this STAGE needs no GPU")

if NEEDS_LEAN:
    sh(f"{PY} -m pip install -q -r requirements-verify.txt")
else:
    print("skipping requirements-verify.txt: this STAGE needs no Lean")

## 3. Mount Drive, restore `results/` and the prebuilt Mathlib

This is what makes the two runtimes one experiment: the GPU box's
`traj_temp*.json` comes back here so the CPU box can verify it.

Restoring the Mathlib tarball skips ~30 minutes of `lake exe cache get` +
`lake build`. Existing local files are never overwritten — a resume log written
during *this* session is newer than whatever is in Drive.

The Lean toolchain setup lives at the end of this cell rather than in cell 2,
because it must run *after* the prebuilt tarball has been restored.

In [ ]:
import glob, os, shutil

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")            # no-op if already mounted
else:
    print("not in Colab -- skipping Drive mount")

os.makedirs("results", exist_ok=True)
os.makedirs("figs", exist_ok=True)

if os.path.isdir(os.path.dirname(DRIVE_DIR)):
    os.makedirs(f"{DRIVE_DIR}/results", exist_ok=True)
    os.makedirs(f"{DRIVE_DIR}/figs", exist_ok=True)

    restored = 0
    for src in glob.glob(f"{DRIVE_DIR}/results/*"):
        dst = os.path.join("results", os.path.basename(src))
        if not os.path.exists(dst):          # never clobber a live local file
            shutil.copy2(src, dst)
            restored += 1
    print(f"restored {restored} file(s) from {DRIVE_DIR}/results")

    tarball = f"{DRIVE_DIR}/{MATHLIB_TARBALL}"
    if NEEDS_LEAN and not os.path.isdir(MATHLIB_DIR) and os.path.exists(tarball):
        print(f"restoring prebuilt Mathlib from {tarball} (skips ~30 min of building)")
        sh(f"tar xzf {tarball} -C .")
else:
    print(f"{DRIVE_DIR} unavailable -- running without Drive backup")

if NEEDS_LEAN:
    # Idempotent: installs elan, clones mathlib4 @ the pinned tag, runs
    # `lake exe cache get` (mandatory -- without it lake compiles Mathlib from
    # source) and `lake build`. Fast once the tarball above is in place.
    sh(f"bash scripts/setup_lean.sh {MATHLIB_DIR}")

## 4. Run the stage

Each temperature is a separate CLI invocation, and each is skipped outright if
its consolidated output already covers `N_QUESTIONS`. Below that, the stages
resume per question from their own `.jsonl`, so an interrupted temperature
restarts at the question it died on.

In [ ]:
import json

def covered(path, n_expected):
    """True if this stage's output already covers every question."""
    if not os.path.exists(path):
        return False
    try:
        with open(path, encoding="utf-8") as fh:
            return json.load(fh).get("n_questions", 0) >= n_expected
    except (json.JSONDecodeError, OSError):
        return False

def run_stage(stage, temp):
    out = ("results/traj_temp{}.json" if stage == "generate"
           else "results/results_temp{}.json").format(fmt_temp(temp))
    if covered(out, N_QUESTIONS):
        print(f"[skip] {stage} temp={fmt_temp(temp)} -- {out} already complete")
        return
    sh(f"{PY} trace_valid.py --stage {stage} --temp {fmt_temp(temp)} "
       f"--n-questions {N_QUESTIONS} --n-traj {N_TRAJ} "
       f"--mathlib-dir {MATHLIB_DIR}")

if STAGE in ("generate", "all"):
    for T in TEMPS:
        run_stage("generate", T)

if STAGE in ("verify", "all"):
    for T in TEMPS:
        run_stage("verify", T)

if STAGE in ("analyze", "all"):
    sweep = " ".join(fmt_temp(t) for t in TEMPS)
    sh(f"{PY} trace_valid.py --stage analyze --sweep {sweep}")

## 5. Back up to Drive

Run this **before** the runtime dies, not after. The Mathlib tarball is written
once and then left alone — it is a build artefact, not a result.

In [ ]:
if os.path.isdir(os.path.dirname(DRIVE_DIR)):
    os.makedirs(f"{DRIVE_DIR}/results", exist_ok=True)
    os.makedirs(f"{DRIVE_DIR}/figs", exist_ok=True)

    n = 0
    for src in glob.glob("results/*") + glob.glob("figs/*"):
        sub = "results" if src.startswith("results") else "figs"
        shutil.copy2(src, f"{DRIVE_DIR}/{sub}/{os.path.basename(src)}")
        n += 1
    print(f"backed up {n} file(s) to {DRIVE_DIR}")

    tarball = f"{DRIVE_DIR}/{MATHLIB_TARBALL}"
    if os.path.isdir(MATHLIB_DIR) and not os.path.exists(tarball):
        print(f"archiving {MATHLIB_DIR} -> {tarball} (a few minutes, once)")
        sh(f"tar czf {tarball} {MATHLIB_DIR}")
    elif os.path.exists(tarball):
        print(f"{tarball} already present -- not rebuilding it")
else:
    print(f"{DRIVE_DIR} unavailable -- nothing backed up")

## 6. Results (analyze only)

Note on the 2×2: under a formal verifier `end_correct` **implies**
`trace_valid`, so the *invalid trace, correct answer* cell is empty by
construction rather than by measurement. That is a design question about the
study, not a bug — see the README, "Two things worth calling out".

In [ ]:
if STAGE in ("analyze", "all"):
    import pandas as pd
    from IPython.display import Image, display

    summary_path = "results/summary.csv"
    if os.path.exists(summary_path):
        summary = pd.read_csv(summary_path)
        display(summary)
        print("\nMathlib:", summary.mathlib_tag.iloc[0],
              "-- Lean results are not comparable across Mathlib versions.")
    else:
        print(f"{summary_path} not found -- run STAGE='analyze' first")

    if os.path.exists("results/crosstab.csv"):
        print("\n2x2: trace_valid x end_correct")
        display(pd.read_csv("results/crosstab.csv", index_col=0))

    for fig in ("figs/temp_sweep.png", "figs/crosstab.png"):
        if os.path.exists(fig):
            print(fig)
            display(Image(fig))
else:
    print(f"STAGE={STAGE} -- set STAGE='analyze' to see the summary and figures")